<a href="https://colab.research.google.com/github/jovanfranco/usdmxn-volatilidad/blob/main/clase18_pruebas_multiples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from scipy import stats
from statsmodels.stats.multitest import multipletests

# 1. Ejemplos a mano
m = 20
print("Prob. de al menos un falso positivo con 20 pruebas:", round(1 - 0.95**m, 3))
print("Umbral de Bonferroni con 20 pruebas:", 0.05 / m)

p = np.array([0.003, 0.011, 0.020, 0.035, 0.300])
for nombre, metodo in [("Bonferroni", "bonferroni"), ("Holm", "holm"), ("Benjamini Hochberg", "fdr_bh")]:
    rechaza, p_aj, _, _ = multipletests(p, alpha=0.05, method=metodo)
    print(nombre, "rechaza:", int(rechaza.sum()), "p ajustados:", p_aj.round(3))

# 2. Cuánto Sharpe sale de puro ruido al probar muchas estrategias
rng = np.random.default_rng(42)
dias, anios = 1500, 1500 / 252
for n in (1, 20, 100):
    mejores = []
    for _ in range(300):
        ruido = rng.normal(0, 0.006, size=(dias, n))
        sharpe = ruido.mean(axis=0) / ruido.std(axis=0, ddof=1) * np.sqrt(252)
        mejores.append(sharpe.max())
    print(f"Mejor Sharpe de puro ruido con {n} estrategias (promedio de 300 repeticiones):", round(float(np.mean(mejores)), 2))
print("Error estándar del Sharpe con", round(anios, 1), "años:", round(1 / np.sqrt(anios), 2))

# 3. Datos reales: USD/MXN
precio = yf.download("MXN=X", start="2020-10-07", end="2026-10-22", progress=False)["Close"].squeeze()
rend = precio.pct_change().dropna()
costo = 0.0001   # 1 punto base por cambio de posición

# 4. Familia de 30 estrategias de cruce de promedios
series = {}
for rapido in (5, 10, 20, 30, 40):
    for lento in (50, 75, 100, 150, 200, 250):
        sen = np.sign(precio.rolling(rapido).mean() - precio.rolling(lento).mean())
        sen = sen.where(precio.rolling(lento).mean().notna())
        pos = sen.shift(1).reindex(rend.index)          # posición de hoy usa información de ayer
        neto = (pos * rend - costo * pos.diff().abs()).dropna()
        series[(rapido, lento)] = neto

inicio = max(s.index[0] for s in series.values())       # mismo periodo para todas
filas = []
for clave, s in series.items():
    s = s.loc[inicio:]
    ins, oos = s.iloc[:int(len(s) * 0.7)], s.iloc[int(len(s) * 0.7):]
    t, pv = stats.ttest_1samp(ins, 0.0)
    filas.append({"rapido": clave[0], "lento": clave[1],
                  "sharpe_in": ins.mean() / ins.std() * np.sqrt(252),
                  "p_in": pv,
                  "sharpe_out": oos.mean() / oos.std() * np.sqrt(252)})
tabla = pd.DataFrame(filas).sort_values("p_in").reset_index(drop=True)
for nombre, metodo in [("bonf", "bonferroni"), ("holm", "holm"), ("bh", "fdr_bh")]:
    tabla[nombre] = multipletests(tabla["p_in"], alpha=0.05, method=metodo)[1]
print(tabla.sort_values("sharpe_in", ascending=False).head(5).round(3).to_string())
print("Pasan al 5% sin corregir:", int((tabla["p_in"] < 0.05).sum()),
      "| Bonferroni:", int((tabla["bonf"] < 0.05).sum()),
      "| Holm:", int((tabla["holm"] < 0.05).sum()),
      "| BH:", int((tabla["bh"] < 0.05).sum()))

# 5. La ganadora dentro de la muestra, ¿qué hace fuera de muestra?
mejor = tabla.sort_values("sharpe_in", ascending=False).iloc[0]
print("Ganadora en muestra:", int(mejor["rapido"]), int(mejor["lento"]),
      "Sharpe dentro:", round(mejor["sharpe_in"], 2), "Sharpe fuera:", round(mejor["sharpe_out"], 2))
print("Sharpe promedio de las 30, dentro y fuera:",
      round(tabla["sharpe_in"].mean(), 2), round(tabla["sharpe_out"].mean(), 2))
print("Estrategias con Sharpe fuera de muestra mayor que cero:", int((tabla["sharpe_out"] > 0).sum()), "de", len(tabla))

Prob. de al menos un falso positivo con 20 pruebas: 0.642
Umbral de Bonferroni con 20 pruebas: 0.0025
Bonferroni rechaza: 1 p ajustados: [0.015 0.055 0.1   0.175 1.   ]
Holm rechaza: 2 p ajustados: [0.015 0.044 0.06  0.07  0.3  ]
Benjamini Hochberg rechaza: 4 p ajustados: [0.015 0.027 0.033 0.044 0.3  ]
Mejor Sharpe de puro ruido con 1 estrategias (promedio de 300 repeticiones): -0.02
Mejor Sharpe de puro ruido con 20 estrategias (promedio de 300 repeticiones): 0.78
Mejor Sharpe de puro ruido con 100 estrategias (promedio de 300 repeticiones): 1.03
Error estándar del Sharpe con 6.0 años: 0.41


/tmp/ipykernel_1027/160145436.py:30: FutureWarning: YF.download() has changed argument auto_adjust default to True
  precio = yf.download("MXN=X", start="2020-10-07", end="2026-10-22", progress=False)["Close"].squeeze()


   rapido  lento  sharpe_in   p_in  sharpe_out  bonf  holm     bh
0      40     75      0.770  0.141       0.514   1.0   1.0  0.995
1      30     75      0.712  0.174       0.221   1.0   1.0  0.995
3      40    100      0.361  0.490       0.822   1.0   1.0  0.995
5      40     50      0.292  0.576       0.046   1.0   1.0  0.995
6      30    100      0.290  0.578       0.955   1.0   1.0  0.995
Pasan al 5% sin corregir: 0 | Bonferroni: 0 | Holm: 0 | BH: 0
Ganadora en muestra: 40 75 Sharpe dentro: 0.77 Sharpe fuera: 0.51
Sharpe promedio de las 30, dentro y fuera: 0.08 0.29
Estrategias con Sharpe fuera de muestra mayor que cero: 25 de 30
